# 🔬 Skin Cancer Detection — Training Notebook

This notebook walks through the full pipeline for training a binary skin lesion classifier:
**Benign vs. Malignant**.

**Approach:** Transfer learning with EfficientNet-B0 pretrained on ImageNet.
We replace the classification head and fine-tune on the ISIC 2017 dataset.

### Sections
1. Setup & Imports
2. Dataset Loading
3. Exploratory Data Analysis
4. Data Preprocessing & Augmentation
5. Model Definition
6. Training — Phase 1 (frozen backbone)
7. Training — Phase 2 (full fine-tuning)
8. Evaluation (accuracy, AUC, confusion matrix, ROC curve)
9. Save Model

## 1. Setup & Imports

In [ ]:
import os
import sys
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image

from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

# Add app/ to path so we can import model.py
sys.path.append(str(Path('../app').resolve()))
from model import build_model, unfreeze_backbone, train_transform, val_transform

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Using device: {DEVICE}')

## 2. Dataset Loading

Download the ISIC 2017 dataset from https://challenge.isic-archive.com/data/#2017

Place files at:
```
data/
├── ISIC-2017_Training_Data/          ← folder of .jpg images
└── ISIC-2017_Training_Part3_GroundTruth.csv
```

The CSV has columns: `image_id`, `melanoma` (1=malignant), `seborrheic_keratosis`.
We treat melanoma=1 as **Malignant** and melanoma=0 as **Benign**.

In [ ]:
DATA_DIR   = Path('../data')
IMAGE_DIR  = DATA_DIR / 'ISIC-2017_Training_Data'
LABELS_CSV = DATA_DIR / 'ISIC-2017_Training_Part3_GroundTruth.csv'

assert IMAGE_DIR.exists(),  f'Image directory not found: {IMAGE_DIR}'
assert LABELS_CSV.exists(), f'Labels CSV not found: {LABELS_CSV}'

df = pd.read_csv(LABELS_CSV)
print(f'Total images: {len(df)}')
print(df.head())

## 3. Exploratory Data Analysis

In [ ]:
# Map to binary label: 1 = Malignant (melanoma), 0 = Benign
df['label'] = df['melanoma'].astype(int)
df['label_name'] = df['label'].map({0: 'Benign', 1: 'Malignant'})

# Class distribution
counts = df['label_name'].value_counts()
print('Class distribution:')
print(counts)
print(f'\nImbalance ratio: {counts["Benign"] / counts["Malignant"]:.1f}:1 (benign:malignant)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(counts.index, counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, (label, count) in enumerate(counts.items()):
    axes[0].text(i, count + 5, str(count), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Class Proportions')

plt.tight_layout()
plt.show()

In [ ]:
# Preview sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Images: Benign (top) | Malignant (bottom)', fontsize=13)

for label, row_axes in zip([0, 1], axes):
    samples = df[df['label'] == label].sample(5, random_state=SEED)
    for ax, (_, row) in zip(row_axes, samples.iterrows()):
        img_path = IMAGE_DIR / f"{row['image_id']}.jpg"
        img = Image.open(img_path)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(row['label_name'], fontsize=9)

plt.tight_layout()
plt.show()

## 4. Data Preprocessing & Augmentation

We split into 80% train / 20% validation, stratified by class.

To handle the ~4:1 class imbalance we use **WeightedRandomSampler** during
training so each batch sees roughly equal benign and malignant examples.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df['label']
)
print(f'Train: {len(train_df)} | Val: {len(val_df)}')
print('Train class counts:', train_df['label_name'].value_counts().to_dict())
print('Val class counts:  ', val_df['label_name'].value_counts().to_dict())

In [ ]:
class SkinLesionDataset(Dataset):
    """
    Custom Dataset for ISIC skin lesion images.
    Loads images on-the-fly from disk and applies the given transform.
    """
    def __init__(self, dataframe: pd.DataFrame, image_dir: Path, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = self.image_dir / f"{row['image_id']}.jpg"
        image    = Image.open(img_path).convert('RGB')
        label    = int(row['label'])

        if self.transform:
            image = self.transform(image)

        return image, label


# ── Datasets ─────────────────────────────────────────────────────────────────
train_dataset = SkinLesionDataset(train_df, IMAGE_DIR, transform=train_transform)
val_dataset   = SkinLesionDataset(val_df,   IMAGE_DIR, transform=val_transform)

# ── Weighted sampler to handle class imbalance ────────────────────────────────
class_counts  = train_df['label'].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = [class_weights[label] for label in train_df['label'].values]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# ── DataLoaders ───────────────────────────────────────────────────────────────
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,   num_workers=2)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
# Visualize augmented training samples
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

def denormalize(tensor):
    """Reverse ImageNet normalization for display."""
    img = tensor.permute(1, 2, 0).numpy()
    img = img * IMAGENET_STD + IMAGENET_MEAN
    return np.clip(img, 0, 1)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle('Augmented Training Samples (top: Benign | bottom: Malignant)', fontsize=11)

label_names = ['Benign', 'Malignant']
for target_label, row_axes in zip([0, 1], axes):
    idxs = [i for i, l in enumerate(labels) if l == target_label][:8]
    for ax, idx in zip(row_axes, idxs):
        ax.imshow(denormalize(images[idx]))
        ax.axis('off')
    for ax in row_axes[len(idxs):]:
        ax.axis('off')

plt.tight_layout()
plt.show()

## 5. Model Definition

We use **EfficientNet-B0** pretrained on ImageNet.

**Why EfficientNet?**
- Strong accuracy-to-parameter ratio
- Widely used in medical imaging tasks
- B0 is fast to train on CPU/laptop GPU

**Two-phase training strategy:**
1. **Phase 1** — Freeze backbone, train only the new classifier head (fast convergence, avoids destroying pretrained features)
2. **Phase 2** — Unfreeze all layers, fine-tune end-to-end at a lower learning rate

In [ ]:
model = build_model(num_classes=2, freeze_backbone=True)
model = model.to(DEVICE)

# Count trainable parameters
total  = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total:,}')
print(f'Trainable parameters: {trainable:,}  ({100*trainable/total:.1f}%)')

## 6. Training — Phase 1 (Frozen Backbone)

In [ ]:
# ── Loss & optimizer ─────────────────────────────────────────────────────────
# Class weights to further penalize missing malignant cases (higher recall)
pos_weight = torch.tensor([class_counts[0] / class_counts[1]]).to(DEVICE)
criterion  = nn.CrossEntropyLoss()

optimizer  = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5, verbose=True)

print('Optimizer and scheduler ready.')

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(loader, leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds      = outputs.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_probs, all_labels = 0.0, [], [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss    = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        probs  = torch.softmax(outputs, dim=1)[:, 1]
        preds  = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    n    = len(all_labels)
    acc  = accuracy_score(all_labels, all_preds)
    auc  = roc_auc_score(all_labels, all_probs)
    return total_loss / n, acc, auc, all_preds, all_probs, all_labels

In [ ]:
# ── Phase 1 training loop ─────────────────────────────────────────────────────
EPOCHS_PHASE1 = 10
SAVE_PATH     = Path('../app/best_model.pth')

best_auc       = 0.0
history        = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_auc': []}

print('=== Phase 1: Training classifier head (backbone frozen) ===')
for epoch in range(1, EPOCHS_PHASE1 + 1):
    train_loss, train_acc                          = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc, val_auc, _, _, _            = evaluate(model, val_loader, criterion, DEVICE)

    scheduler.step(val_auc)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), SAVE_PATH)
        saved = '  ✓ saved'
    else:
        saved = ''

    print(f'Epoch {epoch:2d}/{EPOCHS_PHASE1} | '
          f'Train Loss: {train_loss:.4f}  Acc: {train_acc:.3f} | '
          f'Val Loss: {val_loss:.4f}  Acc: {val_acc:.3f}  AUC: {val_auc:.3f}{saved}')

print(f'\nBest Val AUC (Phase 1): {best_auc:.4f}')

## 7. Training — Phase 2 (Full Fine-Tuning)

In [ ]:
# Load best phase-1 checkpoint and unfreeze
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model = unfreeze_backbone(model)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters after unfreeze: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

# Lower LR for fine-tuning — backbone is fragile, don't destroy pretrained features
optimizer2 = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler2 = optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=10)

EPOCHS_PHASE2 = 10
print(f'\n=== Phase 2: Full fine-tuning for {EPOCHS_PHASE2} epochs ===')

for epoch in range(1, EPOCHS_PHASE2 + 1):
    train_loss, train_acc                          = train_one_epoch(model, train_loader, optimizer2, criterion, DEVICE)
    val_loss, val_acc, val_auc, _, _, _            = evaluate(model, val_loader, criterion, DEVICE)

    scheduler2.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), SAVE_PATH)
        saved = '  ✓ saved'
    else:
        saved = ''

    print(f'Epoch {epoch:2d}/{EPOCHS_PHASE2} | '
          f'Train Loss: {train_loss:.4f}  Acc: {train_acc:.3f} | '
          f'Val Loss: {val_loss:.4f}  Acc: {val_acc:.3f}  AUC: {val_auc:.3f}{saved}')

print(f'\nBest Val AUC overall: {best_auc:.4f}')

## 8. Evaluation

In [ ]:
# Plot training curves
total_epochs = EPOCHS_PHASE1 + EPOCHS_PHASE2
epochs_range = range(1, total_epochs + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Training History', fontsize=13)

axes[0].plot(epochs_range, history['train_loss'], label='Train')
axes[0].plot(epochs_range, history['val_loss'],   label='Val')
axes[0].axvline(EPOCHS_PHASE1 + 0.5, color='gray', linestyle='--', label='Unfreeze')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(epochs_range, history['train_acc'], label='Train')
axes[1].plot(epochs_range, history['val_acc'],   label='Val')
axes[1].axvline(EPOCHS_PHASE1 + 0.5, color='gray', linestyle='--', label='Unfreeze')
axes[1].set_title('Accuracy')
axes[1].legend()

axes[2].plot(epochs_range, history['val_auc'], color='purple')
axes[2].axvline(EPOCHS_PHASE1 + 0.5, color='gray', linestyle='--', label='Unfreeze')
axes[2].set_title('Val AUC-ROC')
axes[2].legend(['AUC', 'Unfreeze'])

plt.tight_layout()
plt.show()

In [ ]:
# Load best checkpoint for final evaluation
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
_, val_acc, val_auc, preds, probs, labels = evaluate(model, val_loader, criterion, DEVICE)

print('=== Final Evaluation on Validation Set ===')
print(f'Accuracy : {val_acc:.4f}')
print(f'AUC-ROC  : {val_auc:.4f}')
print()
print(classification_report(labels, preds, target_names=['Benign', 'Malignant']))

In [ ]:
# Confusion matrix + ROC curve side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
cm = confusion_matrix(labels, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malignant'],
            yticklabels=['Benign', 'Malignant'], ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# ROC curve
fpr, tpr, _ = roc_curve(labels, probs)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {val_auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.savefig('../assets/evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved evaluation plot to assets/evaluation.png')

## 9. Save Model

The best model was already saved to `app/best_model.pth` during training.

Update the README with your final metrics, then run `python app/app.py` to launch the demo.

In [ ]:
print(f'Model saved to: {SAVE_PATH.resolve()}')
print(f'File size: {SAVE_PATH.stat().st_size / 1e6:.1f} MB')
print()
print('Next steps:')
print('  1. Update README.md with your accuracy / AUC numbers')
print('  2. Run: python app/app.py')
print('  3. Deploy to Hugging Face Spaces for a live demo link on your resume')